# 03 — Alıştırma: Seçmeli Ders Seçimi

**Öğrenme hedefleri:**
1. Sözel bir problemi kendi başına ILP modeline çevirmek
2. Greedy (açgözlü) yaklaşımın neden optimal olmayabileceğini görmek
3. Önkoşul ve çakışma kısıtları yazmak
4. İkinci kaynak kısıtı ekleyerek çok boyutlu knapsack'i keşfetmek

**Tahmini süre:** ~25 dakika (15 dk bireysel + 10 dk birlikte çözüm)

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q pulp
    !git clone -q https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import pandas as pd

---
## Problem Tanımı

Bu dönem seçmeli dersler için **14 AKTS**'lik hakkın var.  
Her dersin AKTS değeri ve sana katacağı **fayda puanı** (ilgi + kariyer değeri) aşağıda.  
**Toplam faydayı maksimize edecek dersleri seç.**

| Ders | Ad | AKTS | Fayda | Haftalık Saat |
|------|----|------|-------|---------------|
| A    | Yapay Zekâya Giriş    | 8 | 30 | 4 |
| B    | Bilgisayar Grafikleri | 7 | 20 | 3 |
| C    | Mobil Programlama     | 7 | 20 | 3 |
| D    | Teknik İngilizce      | 4 |  8 | 2 |

In [ ]:
courses = ["A", "B", "C", "D"]
names   = {"A": "Yapay Zekâya Giriş", "B": "Bilgisayar Grafikleri",
           "C": "Mobil Programlama",  "D": "Teknik İngilizce"}
akts    = {"A": 8, "B": 7, "C": 7, "D": 4}
value   = {"A": 30, "B": 20, "C": 20, "D": 8}
hours   = {"A": 4, "B": 3, "C": 3, "D": 2}

AKTS_LIMIT = 14

df = pd.DataFrame({
    "Ders": [f"{k}: {names[k]}" for k in courses],
    "AKTS": [akts[k] for k in courses],
    "Fayda": [value[k] for k in courses],
    "Haftalık Saat": [hours[k] for k in courses],
    "Fayda/AKTS": [round(value[k]/akts[k], 2) for k in courses]
})
print(df.to_string(index=False))

---
## Greedy Yaklaşım

"En verimli dersi al" stratejisi: Fayda/AKTS oranına göre sırala ve kapasiteye sığana kadar seç.

In [ ]:
def greedy_selection(courses, akts, value, limit):
    """Fayda/AKTS oranına göre açgözlü seçim."""
    sorted_courses = sorted(courses, key=lambda c: value[c] / akts[c], reverse=True)
    selected = []
    remaining = limit
    for c in sorted_courses:
        if akts[c] <= remaining:
            selected.append(c)
            remaining -= akts[c]
    return selected

greedy_result = greedy_selection(courses, akts, value, AKTS_LIMIT)
greedy_value  = sum(value[c] for c in greedy_result)
greedy_akts   = sum(akts[c] for c in greedy_result)

print("Greedy seçimi:")
for c in greedy_result:
    print(f"  {c}: {names[c]} — {akts[c]} AKTS, fayda {value[c]}")
print(f"Toplam: {greedy_akts} AKTS, fayda {greedy_value}")

---
## 🔧 Görev: ILP Modelini Kur

Aşağıdaki TODO'ları tamamlayarak problemi ILP olarak çöz.

**İpucu:** Her ders ya seçilir ya seçilmez → binary değişken.

In [ ]:
prob = pulp.LpProblem("course_selection", pulp.LpMaximize)

# TODO: Her ders için binary değişken tanımla
# x = {c: pulp.LpVariable(...) for c in courses}

# TODO: Objective fonksiyonu yaz (toplam faydayı maximize et)
# prob += ...

# TODO: AKTS kısıtını yaz
# prob += ...

# TODO: Çöz
# prob.solve(pulp.PULP_CBC_CMD(msg=0))
# print(f"Durum: {pulp.LpStatus[prob.status]}")

print("TODO'ları tamamlayın!")

---
## Greedy vs ILP Karşılaştırması

*(Modeli çözdükten sonra bu hücreyi çalıştırın)*

In [ ]:
# Bu hücreyi modeli çözdükten sonra çalıştırın
try:
    ilp_result = [c for c in courses if pulp.value(x[c]) > 0.5]
    ilp_value  = sum(value[c] for c in ilp_result)
    ilp_akts   = sum(akts[c] for c in ilp_result)

    print("ILP seçimi:")
    for c in ilp_result:
        print(f"  {c}: {names[c]} — {akts[c]} AKTS, fayda {value[c]}")
    print(f"Toplam: {ilp_akts} AKTS, fayda {ilp_value}")
    print()
    print(f"Greedy fayda : {greedy_value}")
    print(f"ILP fayda    : {ilp_value}")
    print(f"Fark         : {ilp_value - greedy_value}")
    if ilp_value > greedy_value:
        print("=> ILP daha iyi bir çözüm buldu!")
    else:
        print("=> Bu örnekte greedy de optimal sonuca ulaştı.")
except NameError:
    print("Önce ILP modelini çözün.")

---
## Knapsack Bağlantısı

Az önce çözdüğünüz problemin adı: **0-1 Knapsack Problem**  
Her item ya alınır (1) ya alınmaz (0) — bu yüzden binary değişken.

Gerçek hayatta nerede karşılaşırız?
- **Bütçe kısıtıyla proje seçimi** (ROI maks.)
- **Kargo yükleme** (ağırlık kısıtıyla değer maks.)
- **Bulut depolama** (kapasite kısıtıyla veri önceliği)
- **Portföy optimizasyonu** (risk kısıtıyla getiri maks.)

---
## Ek Görevler (Hızlı bitirenler için)

### Ek Görev 1 — Önkoşul Kısıtı

> "Bilgisayar Grafikleri (B) dersini almak için Teknik İngilizce (D) dersini de almalısın."
> 
> **Tahmin:** Bu kısıtla optimal çözüm ne olur?

In [ ]:
# Ek Görev 1: Önkoşul kısıtı
# Önkoşul: x_B <= x_D  (B alınırsa D de alınmış olmalı)

prob_ek1 = pulp.LpProblem("knapsack_prereq", pulp.LpMaximize)
x_ek1 = {c: pulp.LpVariable(f"x_{c}", cat="Binary") for c in courses}

prob_ek1 += pulp.lpSum(value[c] * x_ek1[c] for c in courses)
prob_ek1 += pulp.lpSum(akts[c] * x_ek1[c] for c in courses) <= AKTS_LIMIT
prob_ek1 += x_ek1["B"] <= x_ek1["D"]  # ÖNKOŞUL

prob_ek1.solve(pulp.PULP_CBC_CMD(msg=0))
result_ek1 = [c for c in courses if pulp.value(x_ek1[c]) > 0.5]
print(f"Durum  : {pulp.LpStatus[prob_ek1.status]}")
print(f"Seçilen: {result_ek1}")
print(f"Fayda  : {sum(value[c] for c in result_ek1)}")

> **Eğitmen notu:** B+C = 14 AKTS sığar ama önkoşul gereği B alınırsa D de alınmalı → B+C+D = 18 AKTS > 14. Optimum A+D, fayda **38**.

### Ek Görev 2 — Saat Çakışması

> "Bilgisayar Grafikleri (B) ve Mobil Programlama (C) aynı saatte — ikisi birden alınamaz."
> 
> **Tahmin:** Optimum fayda ne olur?

In [ ]:
prob_ek2 = pulp.LpProblem("knapsack_conflict", pulp.LpMaximize)
x_ek2 = {c: pulp.LpVariable(f"x_{c}", cat="Binary") for c in courses}

prob_ek2 += pulp.lpSum(value[c] * x_ek2[c] for c in courses)
prob_ek2 += pulp.lpSum(akts[c] * x_ek2[c] for c in courses) <= AKTS_LIMIT
prob_ek2 += x_ek2["B"] + x_ek2["C"] <= 1  # ÇAKIŞMA KISITI

prob_ek2.solve(pulp.PULP_CBC_CMD(msg=0))
result_ek2 = [c for c in courses if pulp.value(x_ek2[c]) > 0.5]
print(f"Durum  : {pulp.LpStatus[prob_ek2.status]}")
print(f"Seçilen: {result_ek2}")
print(f"Fayda  : {sum(value[c] for c in result_ek2)}")

> **Eğitmen notu:** B ve C birlikte alınamadığından en iyi seçenek A+D, fayda **38**.

### Ek Görev 3 — İkinci Kaynak: Haftalık Saat Limiti

> Tablodaki haftalık saat sütununu kullan. "Haftada en fazla 5 saat ders" kısıtını ekle.  
> Bu, problemi **çok boyutlu knapsack**'e çevirir.
> 
> **Tahmin:** Hangi dersler seçilir?

In [ ]:
HOUR_LIMIT = 5

prob_ek3 = pulp.LpProblem("knapsack_2d", pulp.LpMaximize)
x_ek3 = {c: pulp.LpVariable(f"x_{c}", cat="Binary") for c in courses}

prob_ek3 += pulp.lpSum(value[c] * x_ek3[c] for c in courses)
prob_ek3 += pulp.lpSum(akts[c]  * x_ek3[c] for c in courses) <= AKTS_LIMIT
prob_ek3 += pulp.lpSum(hours[c] * x_ek3[c] for c in courses) <= HOUR_LIMIT  # YENİ

prob_ek3.solve(pulp.PULP_CBC_CMD(msg=0))
result_ek3 = [c for c in courses if pulp.value(x_ek3[c]) > 0.5]
print(f"Durum  : {pulp.LpStatus[prob_ek3.status]}")
print(f"Seçilen: {result_ek3}")
print(f"Fayda  : {sum(value[c] for c in result_ek3)}")
print(f"Toplam haftalık saat: {sum(hours[c] for c in result_ek3)}")

> **Eğitmen notu:** B+C (6 saat) ve A+D (6 saat) her ikisi de saat limitini aşar. Sadece A (4 saat) sığar, fayda **30**.

### Ek Görev 4 — Ölçek Testi

> 30 derslik rastgele bir örnek üret ve greedy ile ILP'yi karşılaştır.

In [ ]:
import random
import time

random.seed(42)
N = 30
big_courses = list(range(N))
big_akts    = {i: random.randint(2, 10) for i in big_courses}
big_value   = {i: random.randint(5, 50) for i in big_courses}
BIG_LIMIT   = 40

# Greedy
t0 = time.perf_counter()
greedy_big = greedy_selection(big_courses, big_akts, big_value, BIG_LIMIT)
t_greedy = time.perf_counter() - t0

# ILP
t0 = time.perf_counter()
prob_big = pulp.LpProblem("big_knapsack", pulp.LpMaximize)
x_big = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in big_courses}
prob_big += pulp.lpSum(big_value[i] * x_big[i] for i in big_courses)
prob_big += pulp.lpSum(big_akts[i] * x_big[i] for i in big_courses) <= BIG_LIMIT
prob_big.solve(pulp.PULP_CBC_CMD(msg=0))
t_ilp = time.perf_counter() - t0

ilp_big = [i for i in big_courses if pulp.value(x_big[i]) > 0.5]

print(f"Greedy fayda : {sum(big_value[i] for i in greedy_big):4d}  ({t_greedy*1000:.1f} ms)")
print(f"ILP fayda    : {int(pulp.value(prob_big.objective)):4d}  ({t_ilp*1000:.1f} ms)")